<a href="https://colab.research.google.com/github/2026-2-Generative-AI-Project/Segmentation-SAM/blob/feature%2Fnukki/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 설치

In [ ]:
!pip install -q timm kornia einops pillow-heif
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git

# 1. 경로와 설정

In [ ]:
import os, glob, json, time, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from scipy import ndimage
from google.colab import drive, userdata

register_heif_opener()   # 아이폰 HEIC
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IN_DIR = f'{BASE}/dataset/images'               # 입력 사진
GT_DIR = f'{BASE}/dataset/gt'                   # 정답 마스크 (사진과 같은 이름의 .png)
SEG_DIR = f'{BASE}/seg_pipeline'        # 이 노트북의 결과
RMBG_DIR = f'{SEG_DIR}/rmbg_handoff'    # RMBG 1차 마스크 + 박스 → 파인튜닝 담당자에게 전달
W_DIR = f'{BASE}/weights'               # 파인튜닝된 SAM 가중치를 놓는 곳
for d in (IN_DIR, GT_DIR, SEG_DIR, RMBG_DIR, W_DIR):
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

CFG = dict(
    max_side=2048,        # 처리 해상도 (긴 변)
    rmbg_th=0.5,          # RMBG 확률 마스크 → 이진화 기준
    min_area=0.003,       # 사진 넓이 대비 이보다 작은 덩어리는 부스러기로 보고 무시
    box_pad=0.05,         # 박스를 사방으로 늘리는 비율 (RMBG가 덜 잡은 가장자리 보완)
    max_boxes=15,         # 박스 최대 개수 (큰 덩어리부터)
    sam_model='facebook/sam2.1-hiera-small',
    sam_ft_weights=f'{W_DIR}/sam2_ft.pt',   # 파인튜닝 결과. 없으면 B3 건너뜀
    eval_side=1024,       # 평가 해상도 (긴 변)
    bf_tol=0.0075,        # Boundary F1 허용 거리 (사진 대각선 길이 대비)
    test_ratio=0.3,       # 시험용 비율
    seed=42,
)
CATEGORIES = ['normal', 'white', 'multi', 'hand_tray']   # 일반 / 흰 음식 / 여러 음식 / 손·쟁반
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')


def list_images():
    return sorted(p for p in glob.glob(f'{IN_DIR}/*') if p.lower().endswith(EXTS))


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def load_image(p, max_side=CFG['max_side']):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


print('device:', device, '| 사진', len(list_images()), '장 | 정답', len(glob.glob(f'{GT_DIR}/*.png')), '장')

# 2. 사진 분류와 학습/테스트 나누기
* 처음 실행하면 meta.csv가 만들어지고, 드라이브에서 열어 category 칸을 normal, white, multi, hand_tray 중 하나로 고친 뒤 이 셀을 다시 실행
* split.csv는 한번 만들어지면 절대 바뀌지 않음. 파인튜닝 전후 비교를 위한 값
* 나중에 정답이 추가된 사진은 자동으로 학습용으로 들어감

In [ ]:
META = f'{BASE}/dataset/meta.csv'
SPLIT = f'{BASE}/dataset/split.csv'

names = [stem(p) for p in list_images()]
meta = pd.read_csv(META) if os.path.exists(META) else pd.DataFrame(columns=['image', 'category'])
new = [n for n in names if n not in set(meta.image)]
if new:
    meta = pd.concat([meta, pd.DataFrame({'image': new, 'category': 'normal'})], ignore_index=True)
    meta.to_csv(META, index=False)
    print(f'meta.csv에 {len(new)}장 추가 (category=normal). 드라이브에서 열어 분류를 고친 뒤 이 셀을 다시 실행하세요.')
bad = meta[~meta.category.isin(CATEGORIES)]
assert bad.empty, f'category 오타: {bad.to_dict("records")}'

# 정답이 있는 사진만 나눔. 처음 한 번은 분류별로 같은 비율이 되게 나눔
has_gt = {stem(p) for p in glob.glob(f'{GT_DIR}/*.png')}
split = pd.read_csv(SPLIT) if os.path.exists(SPLIT) else pd.DataFrame(columns=['image', 'split'])
todo = meta[meta.image.isin(has_gt) & ~meta.image.isin(set(split.image))]
if len(todo):
    rows = []
    if split.empty:
        rng = random.Random(CFG['seed'])
        for cat, g in todo.groupby('category'):
            imgs = sorted(g.image)
            rng.shuffle(imgs)
            n_test = round(len(imgs) * CFG['test_ratio']) if len(imgs) >= 2 else 0
            rows += [{'image': n, 'split': 'test' if i < n_test else 'train'} for i, n in enumerate(imgs)]
    else:
        rows = [{'image': n, 'split': 'train'} for n in todo.image]
    split = pd.concat([split, pd.DataFrame(rows)], ignore_index=True)
    split.to_csv(SPLIT, index=False)
print(split.merge(meta, on='image').groupby(['split', 'category']).size().unstack(fill_value=0))

In [ ]:
# meta.csv 분류 수정: 바꿀 사진만 적으면 됨 (적지 않은 사진은 그대로)
fix = {
    # hand_tray: 손에 든 음식, 쟁반 위
    'IMG_0405': 'hand_tray',   # 샐러드볼, 초록 쟁반
    'IMG_1871': 'hand_tray',   # 라멘, 나무 쟁반
    'IMG_2118': 'hand_tray',   # 디저트 + 음료, 흰 쟁반
    'IMG_2808': 'hand_tray',   # 음료만 (아래 참고)
    'IMG_3225': 'hand_tray',   # 망고빙수, 검은 쟁반
    'IMG_4478': 'hand_tray',   # 손에 든 크레페
    'IMG_4826': 'hand_tray',   # 손에 든 버거
    'IMG_6696': 'hand_tray',   # 빵 접시, 쟁반
    'IMG_7144': 'hand_tray',   # 케이크 두 접시, 쟁반
    'IMG_7652': 'hand_tray',   # 빙수 + 토스트, 쟁반
    'IMG_9902': 'hand_tray',   # 디저트 그릇, 초록 쟁반
    'SAM_0431': 'hand_tray',   # 와플, 쟁반
    # multi: 음식 그릇 여러 개
    'IMG_0448': 'multi',  'IMG_0463': 'multi',  'IMG_1190': 'multi',  'IMG_1435': 'multi',
    'IMG_2184': 'multi',  'IMG_2904': 'multi',  'IMG_3851': 'multi',  'IMG_5150': 'multi',
    'IMG_5183': 'multi',  'IMG_6063': 'multi',  'IMG_6594': 'multi',  'IMG_6610': 'multi',
    'IMG_6942': 'multi',  'IMG_7142': 'multi',  'IMG_7146': 'multi',  'IMG_7358': 'multi',
    'SAM_0319': 'multi',
    # white: 흰 그릇 + 흰 음식
    'IMG_6644': 'white',       # 치즈케이크, 흰 접시
    'IMG_8034': 'white',       # 흰 면 위 망고, 흰 그릇
    'IMG_1436': 'white'
}
meta = pd.read_csv(META)
sel = meta.image.isin(fix)
meta.loc[sel, 'category'] = meta.loc[sel, 'image'].map(fix)
assert meta.category.isin(CATEGORIES).all(), '철자 확인: normal / white / multi / hand_tray'
meta.to_csv(META, index=False)
print(meta.category.value_counts())

# 3. 모델 불러오기
* 허깅페이스 토큰은 코랩 왼쪽에 HF_TOKEN이라는 이름으로 저장해 두면 자동으로 쓰임

In [ ]:
from huggingface_hub import login
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from sam2.sam2_image_predictor import SAM2ImagePredictor

try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

# RMBG-2.0
rmbg = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True).to(device).eval()
rmbg_tf = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


@torch.no_grad()
def rmbg_mask(img):
    """RMBG-2.0 확률 마스크 (H, W) 0~1"""
    x = rmbg_tf(img).unsqueeze(0).to(device, next(rmbg.parameters()).dtype)
    pred = rmbg(x)[-1].sigmoid()[0, 0].float().cpu()
    pil = transforms.functional.to_pil_image(pred).resize(img.size, Image.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255


# SAM 2.1 원본 (B2)
sam_base = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)

# SAM 2.1 파인튜닝 (B3): 가중치 파일이 있을 때만
sam_ft = None
if os.path.exists(CFG['sam_ft_weights']):
    sam_ft = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
    sd = torch.load(CFG['sam_ft_weights'], map_location=device)
    res = sam_ft.model.load_state_dict(sd, strict=False)
    print('파인튜닝 가중치 불러옴 | 없는 키', len(res.missing_keys), '| 남는 키', len(res.unexpected_keys))
else:
    print('파인튜닝 가중치 없음 → B3 조건은 건너뜀:', CFG['sam_ft_weights'])

# 4. 파이프라인 함수


In [ ]:
def boxes_from_mask(m_bin, cfg=CFG):
    """RMBG 이진 마스크 → 덩어리마다 박스 [x0, y0, x1, y1] (큰 덩어리부터)"""
    H, W = m_bin.shape
    lab, n = ndimage.label(m_bin)
    if n == 0:
        return []
    sizes = ndimage.sum(m_bin, lab, range(1, n + 1))
    cand = []
    for i, sl in enumerate(ndimage.find_objects(lab)):
        if sizes[i] < cfg['min_area'] * H * W:
            continue
        y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        ph, pw = (y1 - y0) * cfg['box_pad'], (x1 - x0) * cfg['box_pad']
        cand.append((sizes[i], [max(0, x0 - pw), max(0, y0 - ph), min(W, x1 + pw), min(H, y1 + ph)]))
    cand.sort(key=lambda c: -c[0])
    return [[float(v) for v in b] for _, b in cand[:cfg['max_boxes']]]


@torch.no_grad()
def sam_refine(predictor, img_np, boxes):
    """박스마다 SAM 실행 → 박스 밖은 잘라내고 → 합치기"""
    H, W = img_np.shape[:2]
    out = np.zeros((H, W), dtype=bool)
    if not boxes:
        return out
    predictor.set_image(img_np)
    masks, _, _ = predictor.predict(box=np.array(boxes, dtype=np.float32), multimask_output=False)
    masks = np.asarray(masks).reshape(-1, H, W) > 0.5
    for (x0, y0, x1, y1), m in zip(boxes, masks):
        clip = np.zeros_like(m)
        clip[int(y0):int(np.ceil(y1)), int(x0):int(np.ceil(x1))] = True
        out |= m & clip
    return out


def segment(img, mode='rmbg+sam'):
    """통합 함수 (나중에 FastAPI의 pipeline/segment.py로 옮길 부분)
    mode: 'rmbg' | 'rmbg+sam' | 'rmbg+sam_ft'
    반환: (마스크 'L' 이미지, 정보 dict)"""
    soft = rmbg_mask(img)
    boxes = boxes_from_mask(soft > CFG['rmbg_th'])
    if mode == 'rmbg' or not boxes:          # RMBG가 음식을 못 찾으면 RMBG 결과 그대로
        m = soft
    else:
        pred = sam_ft if mode == 'rmbg+sam_ft' else sam_base
        assert pred is not None, '파인튜닝 가중치가 없습니다'
        m = sam_refine(pred, np.array(img), boxes).astype(np.float32)
    mask = Image.fromarray((m * 255).astype(np.uint8))
    return mask, {'boxes': boxes, 'n_boxes': len(boxes), 'no_food': not boxes}

# 5. 사진으로 확인해보기
- 디버깅용, 최종 코드에서는 삭제
- 한 장씩 or 여러 장씩 확인 가능

In [ ]:
# 5-1. 후보 찾기: DINO로 그릇 찾기 → (못 찾으면) RMBG 대체 → SAM으로 그릇마다 따기
# 흐름: DINO 그릇 후보 → 없으면 RMBG 덩어리 박스(사진 90% 넘는 박스 제외) → 그것도 없으면 사용자에게 직접 표시 요청
# 3번(모델 불러오기), 4번(파이프라인 함수) 셀을 먼저 실행해야 함. 드라이브에 아무것도 저장하지 않음
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from torchvision.ops import nms

GD_ID = 'IDEA-Research/grounding-dino-tiny'
gd_proc = AutoProcessor.from_pretrained(GD_ID)
gd = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(device).eval()

CAND = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 표시 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식 (그릇과 겹치지 않을 때만 후보로 남김)
    utensil_words=['spoon', 'chopsticks'],                  # 그릇에 걸치면 그 그릇에 합침 (LABELING.md 규칙)
    box_th=0.3,                   # DINO 점수 기준 (단어별 확신도 중 가장 높은 값)
    nms_iou=0.6,                  # 같은 물체에 겹친 박스 정리
    max_area=0.9,                 # 사진의 90% 넘는 박스는 버림 (테이블, 배경)
    food_max_area=0.5,            # food 박스는 사진의 50%까지만 (상차림 전체를 감싸는 박스 방지)
    mm_score=0.6,                 # SAM 후보 마스크 3개 중 점수가 최고점의 60% 이상인 것만 고려
    mm_inside=0.9,                # 그중 90% 이상이 박스(조금 넓힌) 안에 있는 것 중 가장 큰 것 = 전체 물체
    sam_chunk=4,                  # SAM을 박스 몇 개씩 나눠 돌릴지 (메모리 절약)
    utensil_overlap=0.2,          # 수저 마스크의 20% 이상이 그릇 외곽 안에 있으면 걸친 것으로 봄
    part_min=0.1,                 # 그릇 마스크 조각 중 가장 큰 조각의 10% 이상인 것만 남김
    solidity=0.9,                 # 그릇 마스크가 볼록 외곽의 90% 미만이면 (테두리만 남음 등) 외곽으로 채움 (박스 안에서만)
    inside_drop=0.9,              # 그릇 후보가 다른 그릇 안에 90% 이상 들어가면 중복(그릇 위 음식)으로 보고 제거
    extra_max_area=0.4,           # (보완 모드) 그릇 밖 음식 후보의 최대 크기
)
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def is_utensil(label):
    return label in CAND['utensil_words']


try:
    import cv2
except ImportError:
    cv2 = None


def _hull_crop(c):
    if cv2 is None:   # opencv가 없으면 박스로 대신
        return np.ones_like(c)
    pts = cv2.findNonZero(c.astype(np.uint8))
    out = np.zeros(c.shape, np.uint8)
    cv2.fillConvexPoly(out, cv2.convexHull(pts), 1)
    return out.astype(bool)


def local(fn, m, pad=0):
    """마스크가 있는 영역만 잘라서 계산 (사진 전체로 계산하면 매우 느림)"""
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def hull(m):
    """마스크의 볼록 외곽 (그릇을 고무줄로 감싼 영역)"""
    return local(_hull_crop, m)


def box_mask(shape, box, expand=0.1):
    """박스(조금 넓힌) 영역 마스크"""
    H, W = shape
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def keep_parts(c, box):
    """조각 정리: 가장 큰 조각의 10% 이상이고 박스(조금 넓힌) 안에 절반 이상 있는 조각만 남김
    (수저가 그릇을 가로질러 갈라진 조각은 남기고, 옆 그릇에서 묻어 온 조각은 버림)"""
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def clean_dish(m, box):
    """그릇 마스크 정리: 조각 정리 → 틈 메우기 → 구멍 채우기 → 그래도 많이 비어 있으면 볼록 외곽으로
    (외곽으로 채우는 부분은 DINO 박스 안으로만 제한해서 옆 그릇으로 번지지 않게 함)"""
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - 4, 0), min(ys[-1] + 5, m.shape[0])
    x0, x1 = max(xs[0] - 4, 0), min(xs[-1] + 5, m.shape[1])
    c = keep_parts(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3))
    h = hull(out) & (box_mask(m.shape, box, 0.05) | out)
    if h.any() and out.sum() < CAND['solidity'] * h.sum():
        out = h
    return out


def keep_parts_full(m, box):
    """keep_parts를 사진 전체 마스크에 (마스크가 있는 영역만 잘라서) 적용"""
    if not m.any():
        return m
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    y0, y1, x0, x1 = ys[0], ys[-1] + 1, xs[0], xs[-1] + 1
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = keep_parts(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def close_fill(m):
    return local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """DINO: 글자로 물체 찾기 → [{'box', 'label', 'score', 'kind', 'dish'}]  kind: dish / other / food
    박스마다 단어별 확신도를 직접 계산해서 가장 높은 단어로 이름을 붙임 (예: 'bowl pot cup glass' 같은 섞인 이름 방지)"""
    words = CAND['dish_words'] + CAND['other_words'] + CAND['food_words']
    nd, no = len(CAND['dish_words']), len(CAND['other_words'])
    kinds = ['dish'] * nd + ['other'] * no + ['food'] * len(CAND['food_words'])
    text = '. '.join(words) + '.'
    inputs = gd_proc(images=img, text=text, return_tensors='pt').to(device)
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()            # (박스 후보 수, 글자 토큰 수)
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:                                          # 단어가 차지하는 토큰 위치
        sub = gd_proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    word_p = torch.stack(cols, 1)                            # (박스 후보 수, 단어 수)
    score, best = word_p.max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > CAND['box_th'] and box_area(boxes[i].tolist()) <
            (CAND['food_max_area'] if kinds[int(best[i])] == 'food' else CAND['max_area']) * W * H]  # 테이블 전체 같은 박스 제외
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], CAND['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]


def rmbg_boxes(soft, W, H):
    """RMBG 덩어리 박스 중 사진 90% 넘는 것은 버림"""
    return [b for b in boxes_from_mask(soft > CFG['rmbg_th']) if box_area(b) < CAND['max_area'] * W * H]


def pick_whole(ms, ss, box):
    """SAM이 주는 후보 마스크 3개(부분 ~ 전체) 중, 점수가 충분하고 박스 안에 들어오는 것 중 가장 큰 것 = 전체 물체"""
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= CAND['mm_score'] * ss.max()]
    bm = box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= CAND['mm_inside'] * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


@torch.no_grad()
def sam_masks(predictor, img_np, boxes, kinds=None, multimask=True):
    """박스마다 SAM. 박스 밖을 자르지 않고, 합치지 않음 (set_image는 사진당 한 번)
    원본 SAM(multimask=True): 그릇, 음식은 후보 마스크 3개 중 전체 물체를 고르고, 컵, 수저는 점수가 가장 높은 것
    파인튜닝 SAM(multimask=False): 학습한 단일 마스크 출력을 그대로 씀"""
    predictor.set_image(img_np)
    H, W = img_np.shape[:2]
    kinds = kinds or ['dish'] * len(boxes)
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = predictor.predict(box=b, multimask_output=multimask)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multimask:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            kind = kinds[s0 + k]
            out.append(m[k, int(sc[k].argmax())] if kind == 'other' else pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def tidy(cands):
    """그릇 모양 정리(음식 채우기) → 그릇 위 음식 중복 제거 → 그릇 없는 음식 정리 → 걸친 수저 합치기"""
    for c in cands:
        if c['dish'] and c.get('kind') != 'food':
            c['mask'] = clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c.get('kind') != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if big['dish'] and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    dish_union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        dish_union |= d['mask']
    for f in [c for c in cands if c.get('kind') == 'food']:     # 그릇 없이 놓인 음식만 남김
        f['mask'] = close_fill(keep_parts_full(f['mask'], f['box']))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & dish_union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if is_utensil(u['label']) and u['mask'].any()]
    if utensils:
        # SAM은 그릇을 딸 때 수저 자리를 파내므로, 그릇 마스크 대신 그릇의 볼록 외곽과 겹침을 봄
        hulls = {c['id']: hull(c['mask']) for c in cands if c['dish'] and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = close_fill(d['mask'] | u['mask'])   # 손잡이 포함 + 틈 메우기
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


def segment_candidates(img, predictor=None, supplement=False, multimask=None):
    """사진 → 그릇 후보 목록 (서비스의 find_candidates 자리)
    반환: (후보 목록, 정보). 후보 = {'id', 'box', 'label', 'score', 'source', 'dish'(기본 선택), 'mask'}
    supplement=True면 그릇 밖에 따로 놓인 음식을 RMBG로 보완 (항상 RMBG를 돌려서 느려짐, 실험용)"""
    predictor = predictor or sam_base
    if multimask is None:   # 원본 SAM이면 후보 3개 중 고르기, 파인튜닝 SAM이면 단일 마스크
        multimask = predictor is sam_base
    base = np.array(img)
    W, H = img.size
    info, t0 = {'source': 'dino'}, time.perf_counter()

    found = dino_boxes(img)
    info['t_dino'] = time.perf_counter() - t0
    for f in found:
        f['source'] = 'dino'

    soft = None
    if not any(f['dish'] for f in found):            # 그릇도 음식도 못 찾음 → RMBG 대체
        t1 = time.perf_counter()
        soft = rmbg_mask(img)
        info['t_rmbg'] = time.perf_counter() - t1
        rb = rmbg_boxes(soft, W, H)
        found += [{'box': b, 'label': 'food', 'score': 0.0, 'source': 'rmbg', 'kind': 'dish', 'dish': True} for b in rb]
        info['source'] = 'rmbg' if rb else 'none'

    if not found:
        info['message'] = '그릇을 찾지 못했어요. 음식을 직접 표시해 주세요.'
        return [], info

    t2 = time.perf_counter()
    for i, (f, m) in enumerate(zip(found, sam_masks(predictor, base, [f['box'] for f in found], [f['kind'] for f in found], multimask))):
        f.update(id=i, mask=m)
    info['t_sam'] = time.perf_counter() - t2
    t3 = time.perf_counter()
    cands = tidy(found)
    info['t_post'] = time.perf_counter() - t3

    if supplement:                                    # (실험) 그릇 밖 음식 보완
        if soft is None:
            soft = rmbg_mask(img)
        covered = np.zeros((H, W), bool)
        for c in cands:
            covered |= c['mask']
        extra = [b for b in boxes_from_mask((soft > CFG['rmbg_th']) & ~covered)
                 if box_area(b) < CAND['extra_max_area'] * W * H]
        if extra:
            nxt = max(c['id'] for c in cands) + 1
            for j, (b, m) in enumerate(zip(extra, sam_masks(predictor, base, extra, multimask=multimask))):
                cands.append({'id': nxt + j, 'box': b, 'label': 'food', 'score': 0.0,
                              'source': 'rmbg_extra', 'kind': 'food', 'dish': True, 'mask': close_fill(m)})
    info['t_total'] = time.perf_counter() - t0
    return cands, info


def show_candidates(name, supplement=False):
    p = [q for q in list_images() if stem(q) == name][0]
    img = load_image(p)
    base = np.array(img)
    cands, info = segment_candidates(img, supplement=supplement)
    times = ' | '.join(f'{k[2:]} {v:.2f}초' for k, v in info.items() if k.startswith('t_'))
    print(f"{name}: 후보 {len(cands)}개 (그릇 {sum(c['dish'] for c in cands)}) | 박스 출처 {info['source']} | {times}")
    if 'message' in info:
        print(info['message'])
        return cands

    fig, ax = plt.subplots(1, 3, figsize=(18, 7))
    fig.suptitle(f"{name}  |  candidates {len(cands)}, dishes {sum(c['dish'] for c in cands)}  |  {info['t_total']:.2f}s",
                 fontsize=15, fontweight='bold')
    ax[0].imshow(img); ax[0].set_title(f'{name}: candidates ({len(cands)})  green=dish, orange=other')
    over, union = base.copy(), np.zeros(base.shape[:2], bool)
    for k, c in enumerate(cands):
        x0, y0, x1, y1 = c['box']
        col = 'lime' if c['dish'] else 'orange'
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=col, lw=2))
        ax[0].text(x0, y0, f"{c['id']}:{c['label']} {c['score']:.2f}", color='k', fontsize=8, backgroundcolor=col)
        if c['dish']:
            over[c['mask']] = (over[c['mask']] * 0.45 + COLORS[k % len(COLORS)] * 0.55).astype(np.uint8)
            union |= c['mask']
    ax[1].imshow(over); ax[1].set_title('dish candidates (one color = one dish)')
    ax[2].imshow(union, cmap='gray'); ax[2].set_title('dishes only (default selection)')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()
    return cands


def show_many(names, supplement=False):
    """여러 장을 한 번에 확인. 예: show_many(['IMG_2184', 'IMG_6594'])  /  전체: show_many([stem(p) for p in list_images()])"""
    for n in names:
        show_candidates(n, supplement=supplement)


def show_all(cols=4, rows=3, side=600, with_candidates=False):
    """전체 사진을 이름, 현재 분류(meta.csv)와 함께 격자로 보기 (분류 수정용, 빠름)
    with_candidates=True면 사진마다 후보 결과까지 보여 줌 (느림)"""
    paths = list_images()
    if with_candidates:
        return show_many([stem(p) for p in paths])
    meta_path = globals().get('META')
    cat = {}
    if meta_path and os.path.exists(meta_path):
        m = pd.read_csv(meta_path)
        cat = dict(zip(m.image, m.category))
    per = cols * rows
    for s in range(0, len(paths), per):
        chunk = paths[s:s + per]
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.6 * rows), squeeze=False)
        for a in axes.ravel():
            a.axis('off')
        for a, p in zip(axes.ravel(), chunk):
            n = stem(p)
            a.imshow(load_image(p, side))
            a.set_title(f'{n}\n[{cat.get(n, "?")}]', fontsize=11)
        fig.suptitle(f'{s + 1}~{s + len(chunk)} / {len(paths)}', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()


cands = show_candidates('IMG_2184')   # 다른 사진: show_candidates('IMG_6594'), 여러 장: show_many(['IMG_2184', 'IMG_6594'])
# 분류 수정용 전체 보기: show_all()   /   전체 후보 결과까지: show_all(with_candidates=True)

# 6. 전체 사진 실행

In [ ]:
# 6. 전체 사진 실행: 후보 찾기 방식 비교
# B1: RMBG만 / B2: RMBG 박스 → SAM (예전 방식) / C2: DINO → SAM / C3: DINO → (못 찾으면 RMBG) → SAM
# 파인튜닝 가중치(weights/sam2_ft.pt)가 있으면 B3, C3_ft(C3와 같은 방식 + 파인튜닝 SAM)도 자동으로 추가
# C2, C3는 그릇으로 고른 후보만 합친 마스크로 저장 (사용자가 기본 선택을 그대로 둔 경우)
# 5-1번 셀을 먼저 실행해야 함 (segment_candidates 사용)
CONDS = {'B1_rmbg': None, 'B2_rmbg_sam': sam_base, 'C2_dino_sam': 'dino', 'C3_dino_rmbg_sam': 'dino+rmbg'}
if sam_ft is not None:   # 파인튜닝 가중치가 있으면 파인튜닝 SAM 조건도 추가
    CONDS['B3_rmbg_sam_ft'] = sam_ft
    CONDS['C3_dino_rmbg_sam_ft'] = 'dino+rmbg+ft'
DINO_DIR = f'{SEG_DIR}/dino_handoff'     # DINO 후보 박스 → 파인튜닝(2번 노트북)용
for c in CONDS:
    os.makedirs(f'{SEG_DIR}/{c}', exist_ok=True)
os.makedirs(DINO_DIR, exist_ok=True)

handoff, dino_handoff, times = {}, {}, []
for i, p in enumerate(list_images()):
    name, img = stem(p), load_image(p)
    base = np.array(img)
    W, H = img.size

    # RMBG (B1, B2용)
    t0 = time.perf_counter()
    soft = rmbg_mask(img)
    boxes = boxes_from_mask(soft > CFG['rmbg_th'])
    t_rmbg = time.perf_counter() - t0
    Image.fromarray((soft * 255).astype(np.uint8)).save(f'{RMBG_DIR}/{name}.png')
    handoff[name] = {'size': [W, H], 'boxes': boxes,
                     'boxes_norm': [[b[0] / W, b[1] / H, b[2] / W, b[3] / H] for b in boxes]}

    # DINO 방식은 한 번만 실행해서 C2, C3를 같이 만듦
    cands, info = segment_candidates(img)
    dishes = [c for c in cands if c['dish']]
    dish_union = np.zeros((H, W), bool)
    for c in dishes:
        dish_union |= c['mask']
    if sam_ft is not None:
        cands_ft, info_ft = segment_candidates(img, predictor=sam_ft)
        dishes_ft = [c for c in cands_ft if c['dish']]
        union_ft = np.zeros((H, W), bool)
        for c in dishes_ft:
            union_ft |= c['mask']
    dino_handoff[name] = {'size': [W, H], 'source': info['source'],
                          'candidates': [{k: c[k] for k in ('box', 'label', 'score', 'kind', 'source', 'dish')} for c in cands]}

    for c, pred in CONDS.items():
        t1 = time.perf_counter()
        if pred is None:                       # B1
            m, sec, nb = soft, t_rmbg, len(boxes)
        elif pred == 'dino':                   # C2: DINO가 그릇을 못 찾으면 빈 마스크
            m = dish_union if info['source'] == 'dino' else np.zeros((H, W), bool)
            sec, nb = info['t_total'], len(dishes) if info['source'] == 'dino' else 0
        elif pred == 'dino+rmbg':              # C3
            m, sec, nb = dish_union, info['t_total'], len(dishes)
        elif pred == 'dino+rmbg+ft':           # C3 + 파인튜닝 SAM
            m, sec, nb = union_ft, info_ft['t_total'], len(dishes_ft)
        else:                                  # B2, B3: 예전 방식 (RMBG 박스 → SAM)
            m = soft if not boxes else sam_refine(pred, base, boxes)
            sec, nb = t_rmbg + time.perf_counter() - t1, len(boxes)
        Image.fromarray((np.asarray(m, np.float32) * 255).astype(np.uint8)).save(f'{SEG_DIR}/{c}/{name}.png')
        times.append({'image': name, 'cond': c, 'n_boxes': nb, 'sec': round(sec, 3)})
    print(f"[{i + 1}] {name}: RMBG 박스 {len(boxes)}개 | 그릇 후보 {len(dishes)}개 (출처 {info['source']})")

json.dump({'cfg': CFG, 'images': handoff}, open(f'{RMBG_DIR}/boxes.json', 'w'), ensure_ascii=False, indent=1)
json.dump({'cand': CAND, 'images': dino_handoff}, open(f'{DINO_DIR}/candidates.json', 'w'), ensure_ascii=False, indent=1)
tdf = pd.DataFrame(times)
tdf.to_csv(f'{SEG_DIR}/seg_time.csv', index=False)
print(tdf.groupby('cond')['sec'].agg(['mean', 'max']).round(3))
print('출처별 사진 수:', pd.Series([v['source'] for v in dino_handoff.values()]).value_counts().to_dict())

# 7. 평가

In [ ]:
def to_eval(mask_img, size):
    return np.asarray(mask_img.convert('L').resize(size, Image.BILINEAR), dtype=np.float32) / 255


def boundary_f1(p, g, tol):
    """경계선끼리 tol 픽셀 이내로 맞는 비율 (경계 품질)"""
    pb = p ^ ndimage.binary_erosion(p)
    gb = g ^ ndimage.binary_erosion(g)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    prec = (ndimage.distance_transform_edt(~gb)[pb] <= tol).mean()
    rec = (ndimage.distance_transform_edt(~pb)[gb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-8))


def metrics(p_soft, g):
    pb = p_soft > 0.5
    union = (pb | g).sum()
    tol = CFG['bf_tol'] * np.hypot(*g.shape)
    return {'IoU': float((pb & g).sum() / union) if union else 1.0,
            'BF': boundary_f1(pb, g, tol),
            'MAE': float(np.abs(p_soft - g).mean())}


rows = []
info = split.merge(meta, on='image')
for _, r in info.iterrows():
    gt_img = Image.open(f'{GT_DIR}/{r.image}.png')
    s = CFG['eval_side'] / max(gt_img.size)
    size = (round(gt_img.width * s), round(gt_img.height * s))
    g = to_eval(gt_img, size) > 0.5
    for c in CONDS:
        mp = f'{SEG_DIR}/{c}/{r.image}.png'
        if os.path.exists(mp):
            rows.append({'image': r.image, 'split': r.split, 'category': r.category, 'cond': c,
                         **metrics(to_eval(Image.open(mp), size), g)})

mdf = pd.DataFrame(rows)
mdf.to_csv(f'{SEG_DIR}/seg_metrics.csv', index=False)
test = mdf[mdf.split == 'test']
print('=== test 전체 ===')
print(test.groupby('cond')[['IoU', 'BF', 'MAE']].mean().round(4))
print('\n=== test 분류별 IoU ===')
print(test.pivot_table(index='category', columns='cond', values='IoU', aggfunc='mean').round(3))
print('\n=== test 분류별 BF ===')
print(test.pivot_table(index='category', columns='cond', values='BF', aggfunc='mean').round(3))
print('\n=== 정답 사진 전체 (아직 학습 전이라 방식 비교용) ===')
print(mdf.groupby('cond')[['IoU', 'BF', 'MAE']].mean().round(4))
print('\n=== 사진별 IoU ===')
print(mdf.pivot_table(index='image', columns='cond', values='IoU').round(3))

# 8. 전후 샘플 그림(보고서용)

In [ ]:
def tint(img_np, m, color=(255, 0, 0)):
    out = img_np.copy()
    out[m] = (out[m] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    return out

def plot_samples(img_names, save_name):
    cols = ['image', 'GT'] + list(CONDS)
    fig, axes = plt.subplots(len(img_names), len(cols),
                             figsize=(3.2 * len(cols), 3.2 * len(img_names)), squeeze=False)
    for r, name in enumerate(img_names):
        p = [q for q in list_images() if stem(q) == name][0]
        img = load_image(p, CFG['eval_side'])
        base = np.array(img)
        g = to_eval(Image.open(f'{GT_DIR}/{name}.png'), img.size) > 0.5
        for c, col in enumerate(cols):
            ax = axes[r][c]
            if col == 'image':
                ax.imshow(img); ax.set_title(name, fontsize=9)
            elif col == 'GT':
                ax.imshow(tint(base, g, (0, 200, 0))); ax.set_title('GT', fontsize=9)
            else:
                m = to_eval(Image.open(f'{SEG_DIR}/{col}/{name}.png'), img.size) > 0.5
                iou = mdf[(mdf.image == name) & (mdf.cond == col)].IoU
                ax.imshow(tint(base, m))
                ax.set_title(f'{col}\nIoU {iou.iloc[0]:.3f}' if len(iou) else col, fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{SEG_DIR}/{save_name}', dpi=120, bbox_inches='tight')
    plt.show()


test_names = sorted(test.image.unique())
plot_samples(test_names[:12], 'samples_test.png')            # 시험용 사진 샘플

worst = test[test.cond == 'C3_dino_rmbg_sam'].sort_values('IoU').image.head(6).tolist()
plot_samples(worst, 'samples_worst.png')                      # 가장 못한 사진 (한계 분석용)

# 9. 최종 출력(VLM으로 넘길 PNG)

In [ ]:
import io
from PIL import ImageFilter

# 파인튜닝 가중치가 있으면 B3, 없으면 B2 결과를 최종으로 사용
FINAL_COND = 'C3_dino_rmbg_sam_ft' if 'C3_dino_rmbg_sam_ft' in CONDS else 'C3_dino_rmbg_sam'
FINAL_DIR = f'{BASE}/final'
os.makedirs(FINAL_DIR, exist_ok=True)


def make_cutout(img, mask, feather=1):
    """배경이 투명한 음식 누끼 (RGBA). feather: 경계를 살짝 부드럽게"""
    a = mask.convert('L')
    if feather:
        a = a.filter(ImageFilter.GaussianBlur(feather))
    out = img.convert('RGBA')
    out.putalpha(a)
    return out


def to_png_bytes(img):
    """FastAPI로 주고받을 때 쓰는 PNG 바이트"""
    buf = io.BytesIO()
    img.save(buf, 'PNG')
    return buf.getvalue()


manifest = {}
for p in list_images():
    name, img = stem(p), load_image(p)
    mask = Image.open(f'{SEG_DIR}/{FINAL_COND}/{name}.png').convert('L')
    cut = make_cutout(img, mask)
    bbox = mask.point(lambda v: 255 if v > 127 else 0).getbbox()   # 음식이 있는 영역

    img.save(f'{FINAL_DIR}/{name}_image.png')      # 원본 (HEIC도 PNG로 변환, 회전 반영)
    mask.save(f'{FINAL_DIR}/{name}_mask.png')      # 흑백 마스크 (흰색 = 음식)
    cut.save(f'{FINAL_DIR}/{name}_cut.png')        # 투명 배경 누끼
    if bbox:
        cut.crop(bbox).save(f'{FINAL_DIR}/{name}_cut_crop.png')   # 음식 부분만 잘라낸 누끼

    manifest[name] = {
        'image': f'{name}_image.png', 'mask': f'{name}_mask.png',
        'cut': f'{name}_cut.png', 'cut_crop': f'{name}_cut_crop.png' if bbox else None,
        'size': list(img.size), 'food_bbox': list(bbox) if bbox else None,
        'seg_cond': FINAL_COND, 'no_food': bbox is None,
    }

json.dump(manifest, open(f'{FINAL_DIR}/manifest.json', 'w'), ensure_ascii=False, indent=1)
print(f'{len(manifest)}장 저장 → {FINAL_DIR}  (사용한 조건: {FINAL_COND})')
print('음식을 못 찾은 사진:', [n for n, v in manifest.items() if v['no_food']])

# 정답 마스크 만들기

- labeling.ipynb 참고